In [ ]:
pip install -q pandas wandb torch transformers datasets peft accelerate bitsandbytes trl tqdm evaluate bert_score rouge_score nltk sacrebleu codebleu

In [ ]:
# Authenticate locally before training if needed:
# from huggingface_hub import login
# login(token=os.getenv("HF_TOKEN"))

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import gc
import os
import wandb
import evaluate
import numpy as np
import nltk

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("alenphilip/Code-Review-Assistant", split="train")
eval_dataset = load_dataset("alenphilip/Code-Review-Assistant-Eval", split="train")

print("Columns in dataset:", train_dataset.column_names)
print("\nFirst Example Sample:\n", train_dataset[0]["text"])

In [ ]:

gc.collect()
torch.cuda.empty_cache()

In [ ]:

import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
import evaluate

# Ensure memory allocations stay tightly compressed
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
OUTPUT_DIR = "./qwen2.5-3b-sft-qlora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

max_memory_mapping = {0: "13GiB", "cpu": "10GiB"}


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory_mapping,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16,)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "down_proj", "gate_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)

# Metrics evaluation setup
nltk.download('punkt', quiet=True)
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds):
    try:
        predictions, labels = eval_preds
        predicted_ids = predictions

        predicted_ids = np.where(predicted_ids == -100, tokenizer.pad_token_id, predicted_ids)
        labels = np.where(labels == -100, tokenizer.pad_token_id, labels)

        decoded_preds = tokenizer.batch_decode(predicted_ids, skip_special_tokens=False)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=False)

        parsed_preds = []
        parsed_labels = []

        for pred, label in zip(decoded_preds, decoded_labels):
            if "<|im_start|>assistant" in pred:
                assistant_part = pred.split("<|im_start|>assistant")[-1]
                if "<|im_end|>" in assistant_part:
                    assistant_response = assistant_part.split("<|im_end|>")[0].strip()
                    parsed_preds.append(assistant_response)
                else:
                    parsed_preds.append(assistant_part.strip())
            else:
                parsed_preds.append(pred.strip())

            if "<|im_start|>assistant" in label:
                assistant_part = label.split("<|im_start|>assistant")[-1]
                if "<|im_end|>" in assistant_part:
                    assistant_response = assistant_part.split("<|im_end|>")[0].strip()
                    parsed_labels.append(assistant_response)
                else:
                    parsed_labels.append(assistant_part.strip())
            else:
                parsed_labels.append(label.strip())

        parsed_preds = [p for p in parsed_preds if p.strip()]
        parsed_labels = [l for l in parsed_labels if l.strip()]

        if not parsed_preds or not parsed_labels:
            return {"rougeL": 0.0, "bleu": 0.0}

        rouge_results = rouge_metric.compute(predictions=parsed_preds, references=parsed_labels, use_stemmer=True)
        bleu_results = bleu_metric.compute(predictions=parsed_preds, references=[[l] for l in parsed_labels])

        return {
            "rougeL": rouge_results["rougeL"],
            "bleu": bleu_results["bleu"],
        }
    except Exception as e:
        print(f"Metrics parsing fallback triggered. Error details: {e}")
        return {"rougeL": 0.0, "bleu": 0.0}

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    do_train=True,
    do_eval=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    max_length=512,

    fp16=False,
    bf16=True,
    prediction_loss_only=False,

    eval_strategy="steps",
    eval_steps=100,
    eval_accumulation_steps=10,

    learning_rate=3e-4,
    num_train_epochs=1,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",

    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,

    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.05,

    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    dataset_text_field="text",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,

    push_to_hub=True,
    hub_model_id="rose00009/code-reviewer",
    hub_strategy="all_checkpoints",
    hub_always_push=True,

    report_to="wandb"
)

# 7. Start Training
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

print("Starting training securely on L4 GPU...")
print(f"Training on {len(train_dataset)} examples")
print(f"Evaluating on {len(eval_dataset)} examples every {training_args.eval_steps} steps")
print(f"Best model will be selected based on {training_args.metric_for_best_model}")

trainer.train()

print("\n" + "="*70)
print("Training completed!")
print("="*70)

trainer.save_model("./final_code_reviewer_model")
trainer.push_to_hub("Training complete!")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 1. Define your model paths
BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
# This points directly to the repository from your screenshot!
HF_ADAPTER_ID = "rose00009/code-reviewer"

print("Step 1: Downloading and loading the original Qwen 3B Base Model...")
# 2. Load the base model in bfloat16 to match your original L4 training setup
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Step 2: Loading the tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

print("Step 3: Fetching and attaching your custom adapter weights from Hugging Face...")
# 3. This snaps your 74.9 MB adapter files right onto the base model layers
model = PeftModel.from_pretrained(base_model, HF_ADAPTER_ID)

print("🎉 Success! Your custom code-reviewer model is fully assembled and ready to test.")

In [ ]:
# Create a sample chat payload matching your fine-tuning format
messages = [
    {"role": "system", "content": "You are an expert code reviewer."},
    {"role": "user", "content": "Review this Python function:\n\ndef process_data(vals):\n    for i in range(len(vals)):\n        print(vals[i])"}
]

# Format using Qwen's specific template syntax
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")

# Generate response ids
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.4,
        do_sample=True
    )

# Slice out the input tokens to isolate the new assistant response
generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n--- Code Review Output ---")
print(response)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import notebook_login

# 1. Log in with your personal Hugging Face account
print("Log in using your personal WRITE token:")
notebook_login()

# 2. Define your personal repository paths
# Swap out the model ID string below if you are uploading your 3B model variant instead
BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
TARGET_REPO = "rose00009/Code_Review_Assistant_Model"

print(f"Loading {BASE_MODEL_NAME} into memory...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

# 3. Push the structures directly to your username handle
print(f"Uploading model weights to https://huggingface.co/{TARGET_REPO}...")
model.push_to_hub(TARGET_REPO)

print("Uploading tokenizer assets...")
tokenizer.push_to_hub(TARGET_REPO)

print("🎯 Complete! Check your personal Hugging Face dashboard layout now.")